Imports and dates

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Add repo root to PYTHONPATH
HERE = Path().resolve()
PROJECT_ROOT = HERE.parents[0]   # notebooks → repo root
sys.path.insert(0, str(PROJECT_ROOT))


from prm_opt.run_s25 import run_s25_s1, run_s25_s2_v2, run_s25_s2_v2_sensitivity
from prm_opt.run_s26 import run_s26_s1, run_s26_s2_v2, run_s26_s2_v2_sensitivity
from prm_opt.build_s26_assumptions import build_s26_assumptions
from prm_opt.outputs import build_run_report, print_run_report, build_run_report_s1, print_run_report_s1
from prm_opt.build_jobs import build_jobs
from prm_opt.ingest_s25 import ingest_s25
from prm_opt.config import PlanningToggles

# -----------------------------
# DATE RANGES
# -----------------------------
START_S25 = "2025-03-30"
END_S25   = "2025-10-26"

START_S26 = "2026-05-16"
END_S26   = "2026-10-24"

Common toggles


In [ ]:

toggles = PlanningToggles(
    sla_buffer_mins=0,
    spill_bucket_cap=12,
    standby_dep_vert_mins=10,
    standby_arr_horiz_mins=10,
)


Run S25 Scenario 1 (baseline)

In [ ]:

out_s1 = run_s25_s1(START_S25, END_S25, toggles=toggles)

report_s1 = build_run_report_s1(out_s1, day_from="s", hour_method="max")
print_run_report_s1(report_s1)

# # 15-min peak day detail:
# report_s1["peak_day_report"]["bucket_level"].to_csv("S1_peak_day_15min.csv")
# report_s1["peak_day_report"]["hourly"].to_csv("S1_peak_day_hourly.csv")





Unmatched passenger rows after merge (missing Chocks DT): 422
Unique unmatched flight keys: 274

Unmatched key reasons:
reason
no_flight_candidate               213
scheduled_dt_mismatch              60
exact_match_should_have_joined      1
Name: count, dtype: int64

Dropped 420 passenger rows due to reasons {'scheduled_dt_mismatch', 'no_flight_candidate'}

PRM OPT — Scenario 1 (Policy Baseline) Output

[1] Horizon Summary (Peak across horizon; gaps vs CURRENT fleet)
  PeakAmb       : 7
  PeakMini      : 2
  PeakDrivers   : 8
  PeakVehAgents : 8
  CurrentAmb    : 14
  CurrentMini   : 3
  GapAmb        : 0
  GapMini       : 0

[2] Peak Day (most PRM jobs) — Hourly Requirement + Gaps
  Peak day: 2025-04-15 | jobs=503
  Current fleet: Amb=14 | Mini=3
  Peak hour req : Amb=5 | Mini=2 | Drivers=6 | VehAgents=6
  Peak hour gap : Amb=0 | Mini=0

  Hourly table (first rows):
                     Amb_req  Mini_req  Drivers_req  VehAgents_req  Amb_gap  Mini_gap
s                                

C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\GitHub\EDI_airport_analytics\prm_opt\outputs.py:717: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = bucket_level.resample("H").max()


Run S25 Scenario 2 (optimised)

In [ ]:

sens = run_s25_s2_v2_sensitivity(
    start=START_S25,
    end=END_S25,
    vertical_cycle_grid=[10],
    solver_name="highs",
    toggles=toggles,
    time_limit_sec=600,
    threads=8,
    mip_rel_gap=0.20,
)

# Build a comparison table

for r in sens["runs"]:
    print("\n" + "#" * 80)
    print(f"VERTICAL CYCLE SENSITIVITY: {r['vertical_cycle_mins']} mins")
    print("#" * 80)

    report = build_run_report(r)

    # attach assumption for printing (not rebuilding anything)
    report["assumptions"] = {
        "vertical_cycle_mins": r["vertical_cycle_mins"]
    }

    print_run_report(report)



PRM OPT — S25 Scenario 2 (v2) — Sensitivity
Window : 2025-03-30 → 2025-04-26
vertical_cycle_grid: [10]

[1/4] ingest_s25…

Unmatched passenger rows after merge (missing Chocks DT): 422
Unique unmatched flight keys: 274

Unmatched key reasons:
reason
no_flight_candidate               213
scheduled_dt_mismatch              60
exact_match_should_have_joined      1
Name: count, dtype: int64

Dropped 420 passenger rows due to reasons {'scheduled_dt_mismatch', 'no_flight_candidate'}
    ✓ ingest_s25 done | rows=11,130   [15.17s]
[2/4] build_jobs…
    ✓ build_jobs done | jobs=11,130   [0.17s]
[3/4] build_tau + classes + spin_removed…
    ✓ params built | N_AMB=14   [0.23s]
[4/4] pre_solve_debug (no solver)…

PRE-SOLVER FEASIBILITY DEBUG (no solver)
✅ Required columns present.
Missingness: {'sla_start_time': 0, 'Scheduled Flight DT': 0, 'Chocks DT': 2}

Top missing-flight-anchor by Airline Code + dir (top 20):
Airline Code  dir
EI            D      2
dtype: int64

Bucket horizon: start=2025-0

C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\GitHub\EDI_airport_analytics\prm_opt\outputs.py:501: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = bucket_totals.resample("H").max()


Run S26 Scenario 1

In [ ]:
assumptions = build_s26_assumptions(s25_start=START_S25, s25_end=END_S25)


Unmatched passenger rows after merge (missing Chocks DT): 422
Unique unmatched flight keys: 274

Unmatched key reasons:
reason
no_flight_candidate               213
scheduled_dt_mismatch              60
exact_match_should_have_joined      1
Name: count, dtype: int64

Dropped 420 passenger rows due to reasons {'scheduled_dt_mismatch', 'no_flight_candidate'}
[1/5] Penetration + SSR mix…
    ✓ penetration + SSR mix built   [64.99s]
[2/5] Service times…
    ✓ service times built   [0.22s]
[2b/5] Tau mode params (S25 medians)…
    ✓ tau mode params built   [0.00s]
[3/5] Chocks offsets…
    ✓ chocks offsets built   [0.01s]
[4/5] Early/late timing…
    ✓ early/late params built   [0.01s]
[5/5] Stand inputs…
    ✓ stand inputs built   [0.01s]


In [ ]:
out_s26_s1 = run_s26_s1(
    start=START_S26,
    end=END_S26,
    **assumptions["inputs"],
    toggles=PlanningToggles(),
)


report_s26_s1 = build_run_report_s1(out_s26_s1, day_from="s", hour_method="max")
print_run_report_s1(report_s26_s1)


# report_s26_s1["peak_day_report"]["bucket_level"].to_csv("S26_S1_peak_day_15min.csv")
# report_s26_s1["peak_day_report"]["hourly"].to_csv("S26_S1_peak_day_hourly.csv"




[DEBUG A] Future flights loaded & Pax computed
df_flights rows: 5945
Airlines (sample): ['AF' 'KL' 'FR' 'U2' 'LH' 'LS' 'BA' 'RK' 'LM' 'QR']
Countries (sample): ['FRANCE' 'NETHERLANDS' 'SPAIN' 'BULGARIA' 'GREAT BRITAIN' 'POLAND'
 'PORTUGAL' 'ITALY' 'GERMANY' 'TURKEY']

Pax describe:
count    5945.000000
mean      144.803869
std        48.969755
min         0.000000
25%       121.000000
50%       157.000000
75%       177.000000
max       397.000000
Name: Pax, dtype: float64

Top 10 rows (Airline, CountryName, Sector, dir, Pax):
   Airline    CountryName         Sector dir    Pax
0       AF         FRANCE  International   D  118.0
1       KL    NETHERLANDS  International   D  139.0
2       FR          SPAIN  International   D  183.0
3       FR       BULGARIA  International   D  197.0
4       U2  GREAT BRITAIN       Domestic   D  159.0
5       FR         POLAND  International   D  180.0
6       FR          SPAIN  International   D  177.0
7       U2          SPAIN  International   D  156.0

C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\GitHub\EDI_airport_analytics\prm_opt\outputs.py:717: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = bucket_level.resample("H").max()


Run S26 scenario 2

In [ ]:



out_s26_s2 = run_s26_s2_v2_sensitivity(
    start=START_S26,
    end=END_S26,
    **assumptions["inputs"],
    vertical_cycle_grid=(15),
    solver_name="highs",
    toggles=PlanningToggles(),
    time_limit_sec=600,
    threads=8,
    mip_rel_gap=0.20,
)



for r in out_s26_s2["runs"]:
    print("\n" + "#" * 80)
    print(f"VERTICAL CYCLE SENSITIVITY (S26): {r['vertical_cycle_mins']} mins")
    print("#" * 80)

    report = build_run_report(r, day_from="s", hour_method="max")
    report["assumptions"] = {"vertical_cycle_mins": r["vertical_cycle_mins"]}
    print_run_report(report)
